In [1]:
import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import numpy as np

# Add src to path just in case
sys.path.append(os.path.abspath(os.getcwd()))

try:
    from src.models.SlotGated import SlotGated
except ImportError:
    # If standard import fails, try adjusting path or direct import
    from src.models.SlotGated import SlotGated

def parse_line(line):
    utterance_data, intent_label = line.split(" <=> ")
    items = utterance_data.split()
    words = [item.rsplit(':', 1)[0] for item in items]
    word_labels = [item.rsplit(':', 1)[1] for item in items]
    return {
        'intent_label': intent_label,
        'words': words,
        'word_labels': word_labels,
        'length': len(words)
    }

def load_data(path):
    lines = Path(path).read_text('utf-8').strip().splitlines()
    return [parse_line(line) for line in lines]

train_data = load_data('dataset/train')
valid_data = load_data('dataset/valid')
test_data = load_data('dataset/test')

print(f"Train size: {len(train_data)}")
print(f"Valid size: {len(valid_data)}")
print(f"Test size: {len(test_data)}")

Train size: 13084
Valid size: 700
Test size: 700


In [2]:
class Vocabulary:
    def __init__(self):
        self.word2idx = {'<PAD>': 0, '<UNK>': 1}
        self.intent2idx = {}
        self.slot2idx = {'<PAD>': 0}
        
    def build_vocab(self, data):
        for entry in data:
            # Words
            for word in entry['words']:
                if word not in self.word2idx:
                    self.word2idx[word] = len(self.word2idx)
            
            # Intent
            intent = entry['intent_label']
            if intent not in self.intent2idx:
                self.intent2idx[intent] = len(self.intent2idx)
                
            # Slots
            for slot in entry['word_labels']:
                if slot not in self.slot2idx:
                    self.slot2idx[slot] = len(self.slot2idx)
                    
    def __len__(self):
        return len(self.word2idx)

vocab = Vocabulary()
vocab.build_vocab(train_data)

print(f"Vocab Size: {len(vocab.word2idx)}")
print(f"Intent Size: {len(vocab.intent2idx)}")
print(f"Slot Size: {len(vocab.slot2idx)}")

Vocab Size: 13457
Intent Size: 7
Slot Size: 73


In [3]:
class IntentSlotDataset(Dataset):
    def __init__(self, data, vocab):
        self.data = data
        self.vocab = vocab
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        entry = self.data[idx]
        
        words = [self.vocab.word2idx.get(w, self.vocab.word2idx['<UNK>']) for w in entry['words']]
        intent = self.vocab.intent2idx.get(entry['intent_label'])
        slots = [self.vocab.slot2idx.get(s, self.vocab.slot2idx['<PAD>']) for s in entry['word_labels']]
        
        return torch.tensor(words), torch.tensor(intent), torch.tensor(slots)

def collate_fn(batch):
    words, intents, slots = zip(*batch)
    lengths = torch.tensor([len(w) for w in words])
    
    # Pad words and slots
    words_padded = torch.nn.utils.rnn.pad_sequence(words, batch_first=True, padding_value=0)
    slots_padded = torch.nn.utils.rnn.pad_sequence(slots, batch_first=True, padding_value=0)
    intents = torch.stack(intents)
    
    return words_padded, lengths, intents, slots_padded

train_dataset = IntentSlotDataset(train_data, vocab)
valid_dataset = IntentSlotDataset(valid_data, vocab)
test_dataset = IntentSlotDataset(test_data, vocab)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = SlotGated(
    vocab_size=len(vocab.word2idx),
    embedding_dim=128,
    hidden_dim=256,
    slot_dim=len(vocab.slot2idx),
    intent_dim=len(vocab.intent2idx),
    dropout=0.3
).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion_intent = nn.CrossEntropyLoss()
criterion_slot = nn.CrossEntropyLoss(ignore_index=0) # Ignore padding

epochs = 10
best_valid_loss = float('inf')

for epoch in range(epochs):
    model.train()
    total_loss = 0
    intent_acc = 0
    total_samples = 0
    
    for words, lengths, intents, slots in train_loader:
        words = words.to(device)
        intents = intents.to(device)
        slots = slots.to(device)
        # lengths usually needs to be on CPU for pack_padded_sequence in older PyTorch versions
        # but modern versions handle it. SlotGated uses .cpu() internally just in case.
        
        optimizer.zero_grad()
        
        intent_logits, slot_logits = model(words, lengths)
        
        loss_intent = criterion_intent(intent_logits, intents)
        loss_slot = criterion_slot(slot_logits.view(-1, len(vocab.slot2idx)), slots.view(-1))
        
        loss = loss_intent + loss_slot
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        # Simple accuracy tracking for intent
        _, predicted_intent = torch.max(intent_logits, 1)
        intent_acc += (predicted_intent == intents).sum().item()
        total_samples += intents.size(0)
        
    avg_loss = total_loss / len(train_loader)
    train_acc = intent_acc / total_samples
    print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f} | Train Intent Acc: {train_acc:.4f}")
    
    # Validation
    model.eval()
    valid_loss = 0
    correct_intent = 0
    total_valid = 0
    
    with torch.no_grad():
        for words, lengths, intents, slots in valid_loader:
            words = words.to(device)
            intents = intents.to(device)
            slots = slots.to(device)
            
            intent_logits, slot_logits = model(words, lengths)
            
            loss_intent = criterion_intent(intent_logits, intents)
            loss_slot = criterion_slot(slot_logits.view(-1, len(vocab.slot2idx)), slots.view(-1))
            valid_loss += (loss_intent + loss_slot).item()
            
            _, predicted_intent = torch.max(intent_logits, 1)
            correct_intent += (predicted_intent == intents).sum().item()
            total_valid += intents.size(0)
            
    avg_valid_loss = valid_loss / len(valid_loader)
    valid_acc = correct_intent / total_valid
    print(f"       | Valid Loss: {avg_valid_loss:.4f} | Valid Intent Acc: {valid_acc:.4f}")
    
    if avg_valid_loss < best_valid_loss:
        best_valid_loss = avg_valid_loss
        torch.save(model.state_dict(), 'slot_gated_model.pth')
        print("       | Saved Best Model")

Using device: cuda
Epoch 1 | Loss: 1.4341 | Train Intent Acc: 0.9058
       | Valid Loss: 0.5264 | Valid Intent Acc: 0.9686
       | Saved Best Model
Epoch 2 | Loss: 0.4297 | Train Intent Acc: 0.9773
       | Valid Loss: 0.2966 | Valid Intent Acc: 0.9886
       | Saved Best Model
Epoch 3 | Loss: 0.2724 | Train Intent Acc: 0.9847
       | Valid Loss: 0.2440 | Valid Intent Acc: 0.9886
       | Saved Best Model
Epoch 4 | Loss: 0.1870 | Train Intent Acc: 0.9918
       | Valid Loss: 0.2120 | Valid Intent Acc: 0.9886
       | Saved Best Model
Epoch 5 | Loss: 0.1376 | Train Intent Acc: 0.9947
       | Valid Loss: 0.2096 | Valid Intent Acc: 0.9857
       | Saved Best Model
Epoch 6 | Loss: 0.1087 | Train Intent Acc: 0.9956
       | Valid Loss: 0.1884 | Valid Intent Acc: 0.9829
       | Saved Best Model
Epoch 7 | Loss: 0.0811 | Train Intent Acc: 0.9972
       | Valid Loss: 0.1830 | Valid Intent Acc: 0.9857
       | Saved Best Model
Epoch 8 | Loss: 0.0619 | Train Intent Acc: 0.9989
       | Valid

In [7]:
def predict(model, sentence, vocab, device):
    model.eval()
    words = sentence.split()
    word_indices = [vocab.word2idx.get(w, vocab.word2idx['<UNK>']) for w in words]
    
    x = torch.tensor(word_indices).unsqueeze(0).to(device) # [1, seq_len]
    lengths = torch.tensor([len(words)])
    
    with torch.no_grad():
        intent_logits, slot_logits = model(x, lengths)
        
    # Decode Intent
    intent_idx = torch.argmax(intent_logits, dim=1).item()
    intent_label = [k for k, v in vocab.intent2idx.items() if v == intent_idx][0]
    
    # Decode Slots
    slot_indices = torch.argmax(slot_logits, dim=2).squeeze(0).tolist()
    slot_labels = []
    idx2slot = {v: k for k, v in vocab.slot2idx.items()}
    
    for idx in slot_indices:
        slot_labels.append(idx2slot.get(idx, '<UNK>'))
        
    return intent_label, list(zip(words, slot_labels))

# Example usage
sample_sentence = "whats the weather like for the next three days?"
print(f"Sentence: {sample_sentence}")

intent, slots = predict(model, sample_sentence, vocab, device)
print(f"Predicted Intent: {intent}")
print("Slots:")
for word, slot in slots:
    print(f"  {word}: {slot}")

Sentence: whats the weather like for the next three days?
Predicted Intent: GetWeather
Slots:
  whats: O
  the: O
  weather: O
  like: O
  for: O
  the: O
  next: B-timeRange
  three: I-timeRange
  days?: I-timeRange
